In [1]:
import pandas as pd
import numpy as np

# Create synthetic raw customer data with intent to mimic real-world cleaning issues
data = {
    'Customer_ID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 101, 102, 111, 112, 113],
    'Customer_Name': [' John Doe ', 'Jane Smith', 'Alice Johnson', 'Bob Brown', ' Charlie Davis ',
                     'Eva Green', 'Frank Wright', 'Grace Hopper', 'Henry Ford', 'Ivy Chen',
                     ' John Doe ', 'Jane Smith', 'Jack Ryan', 'Karen Gillan', 'Leo Major'],
    'Join_Date': ['2022-01-15', '15/02/2022', '2022.03.10', '2022-04-05', np.nan,
                  '2022-06-20', '07-22-2022', '2022/08/15', '2022-09-01', '2022-10-10',
                  '2022-01-15', '15/02/2022', '2022-11-12', '2022-12-01', '2023-01-05'],
    'City': ['New York', 'Los Angeles', np.nan, 'Chicago', 'Houston',
             'Phoenix', 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas',
             'New York', 'Los Angeles', 'Austin', np.nan, 'San Jose'],
    'Annual_Income': ['$50,000', '$65,000', '$80,000', '$120,000', '$45,000',
                      '$95,000', '$52,000', np.nan, '$71,000', '$88,000',
                      '$50,000', '$65,000', '$1,200,000', '$60,000', '$75,000'],  # Contains symbol issues & extreme outlier ($1.2M)
    'Total_Spend': [1200.5, 3400.0, 1500.25, 8900.0, 450.0,
                    np.nan, 2100.75, 5000.0, 3100.0, 4200.5,
                    1200.5, 3400.0, 25000.0, 1800.0, 2200.0]  # Contains numeric outlier (25000)
}

# Create DataFrame and save as CSV
df_gen = pd.DataFrame(data)
df_gen.to_csv('customer_dataset.csv', index=False)

print("✅ 'customer_dataset.csv' created successfully in your session storage!")

✅ 'customer_dataset.csv' created successfully in your session storage!


In [2]:
# ==============================================================================
# SECTION 1: TITLE & STUDENT DETAILS
# ==============================================================================
"""
Project Title: Data Analysis & Data Cleaning Assignment
Student Name: [Your Full Name]
Roll Number: [Your Roll Number]
Batch: [Your Batch]
Course: BTech Data Science / Computer Science
"""

# ==============================================================================
# SECTION 2: IMPORT LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np

# ==============================================================================
# SECTION 3: LOAD DATASET
# ==============================================================================
# Replace 'customer_dataset.csv' with your actual file path or URL provided by the mentor
file_path = 'customer_dataset.csv'

df_original = pd.read_csv(file_path)
df = df_original.copy() # Working copy to preserve original dataset

# ==============================================================================
# SECTION 4: DATA EXPLORATION
# ==============================================================================
print("--- 1. First 5 Rows ---")
print(df.head())

print("\n--- 2. Last 5 Rows ---")
print(df.tail())

print("\n--- 3. Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- 4. Column Names ---")
print(df.columns.tolist())

print("\n--- 5. Data Types ---")
print(df.dtypes)

print("\n--- 6. Basic Statistical Information ---")
print(df.describe(include='all'))

# ==============================================================================
# SECTION 5: MISSING VALUES
# ==============================================================================
print("\n--- Missing Values Count per Column ---")
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({'Missing Values': missing_count, 'Percentage (%)': missing_percent})
print(missing_df[missing_df['Missing Values'] > 0])

# Handling Missing Values
# 1. Numerical Columns: Fill missing values with Median (robust to outliers)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# 2. Categorical/Text Columns: Fill missing values with Mode or 'Unknown'
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())

# ==============================================================================
# SECTION 6: DUPLICATES
# ==============================================================================
total_duplicates = df.duplicated().sum()
print(f"\nTotal Duplicate Records Found: {total_duplicates}")

if total_duplicates > 0:
    print("\n--- Sample Duplicate Records ---")
    print(df[df.duplicated(keep=False)].head())

    # Remove duplicate records
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\nVerification: Duplicates remaining = {df.duplicated().sum()}")

# ==============================================================================
# SECTION 7: WRONG FORMATS & CONSISTENCY ISSUES
# ==============================================================================
# Format Fix 1: Standardize Date Formats (e.g., 'Date' or 'Join_Date' column)
date_cols = [col for col in df.columns if 'date' in col.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Format Fix 2: Clean Text & Remove Trailing Whitespaces (e.g., 'Customer_Name', 'City')
for col in cat_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip().str.title()

# Format Fix 3: Clean Currency or Numerical Strings (e.g., removing '$' or ',')
for col in df.columns:
    if df[col].dtype == 'object':
        if df[col].str.contains(r'[\$,]', regex=True).any():
            df[col] = df[col].str.replace(r'[\$,]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n--- Data Types After Format Corrections ---")
print(df.dtypes)

# ==============================================================================
# SECTION 8: OUTLIERS (IQR METHOD)
# ==============================================================================
# Target numerical column for outlier analysis (e.g., 'Annual_Income' or 'Total_Spend')
# Dynamically select the first numeric column available
target_num_col = df.select_dtypes(include=[np.number]).columns[0]

Q1 = df[target_num_col].quantile(0.25)
Q3 = df[target_num_col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[target_num_col] < lower_bound) | (df[target_num_col] > upper_bound)]
print(f"\n--- Outlier Detection for Column: '{target_num_col}' ---")
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
print(f"Number of Outliers Detected: {len(outliers)}")

# Handling Outliers: Capping (Winsorizing) to protect real-world variability
df[target_num_col] = np.where(
    df[target_num_col] > upper_bound, upper_bound,
    np.where(df[target_num_col] < lower_bound, lower_bound, df[target_num_col])
)
print("Outliers capped successfully within upper and lower bounds.")

# ==============================================================================
# SECTION 9: NUMPY ANALYSIS
# ==============================================================================
np_data = df[target_num_col].to_numpy()

mean_val = np.mean(np_data)
median_val = np.median(np_data)
min_val = np.min(np_data)
max_val = np.max(np_data)
std_val = np.std(np_data)

print(f"\n--- NumPy Statistical Analysis ({target_num_col}) ---")
print(f"Mean: {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"Minimum Value: {min_val:.2f}")
print(f"Maximum Value: {max_val:.2f}")
print(f"Standard Deviation: {std_val:.2f}")

# ==============================================================================
# SECTION 10: BEFORE VS AFTER COMPARISON
# ==============================================================================
comparison_data = {
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Duplicate Records',
        'Total Missing Values'
    ],
    'Before Cleaning': [
        df_original.shape[0],
        df_original.shape[1],
        df_original.duplicated().sum(),
        df_original.isnull().sum().sum()
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n--- Dataset Comparison Summary ---")
print(df_comparison.to_string(index=False))

--- 1. First 5 Rows ---
   Customer_ID    Customer_Name   Join_Date         City Annual_Income  \
0          101        John Doe   2022-01-15     New York       $50,000   
1          102       Jane Smith  15/02/2022  Los Angeles       $65,000   
2          103    Alice Johnson  2022.03.10          NaN       $80,000   
3          104        Bob Brown  2022-04-05      Chicago      $120,000   
4          105   Charlie Davis          NaN      Houston       $45,000   

   Total_Spend  
0      1200.50  
1      3400.00  
2      1500.25  
3      8900.00  
4       450.00  

--- 2. Last 5 Rows ---
    Customer_ID Customer_Name   Join_Date         City Annual_Income  \
10          101     John Doe   2022-01-15     New York       $50,000   
11          102    Jane Smith  15/02/2022  Los Angeles       $65,000   
12          111     Jack Ryan  2022-11-12       Austin    $1,200,000   
13          112  Karen Gillan  2022-12-01          NaN       $60,000   
14          113     Leo Major  2023-01-05    